# 🔬 Unified SegMoTE vs MoE-Segformer Evaluator (ISIC 2018 Dataset)

This notebook reproduces the exact methodology requested: Training both the SegMoTE foundation model and the lightweight MoE-Segformer baseline on the **ISIC 2018 Task 1 Skin Lesion Segmentation** dataset (the exact out-of-domain evaluation dataset used in the SegMoTE paper).


In [1]:
%cd /content
!rm -rf vision_tranformer_moe
!git clone https://github.com/toqeer-ahmed/vision_tranformer_moe.git
%cd /content/vision_tranformer_moe
!pip install -r requirements.txt
!pip install kaggle albumentations tensorboard transformers torch torchvision pandas

# Run the complete comparison pipeline!
!python scripts/run_unified_comparison.py


/content
Cloning into 'vision_tranformer_moe'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 231 (delta 116), reused 155 (delta 56), pack-reused 0 (from 0)
Receiving objects: 100% (231/231), 11.77 MiB | 28.48 MiB/s, done.
Resolving deltas: 100% (116/116), done.
/content/vision_tranformer_moe
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempt

## 2. Authenticate with Kaggle

In [2]:
import os
# Set Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = "KGAT_f92b021c2b42601bd960c76192014a55"
!mkdir -p ~/.kaggle
!echo "KGAT_f92b021c2b42601bd960c76192014a55" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
print("Kaggle authentication configured!")

Kaggle authentication configured!


In [3]:
# 3. Download and Prepare the ISIC 2018 Dataset
!kaggle datasets download -d tshenq/isic-2018-task-1
!unzip -q isic-2018-task-1.zip -d isic_temp

import os
import shutil
import glob

dest_dir = "data/medical_dataset"
os.makedirs(os.path.join(dest_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(dest_dir, "masks"), exist_ok=True)

image_files = []
for root, dirs, files in os.walk('isic_temp'):
    for file in files:
        if file.lower().endswith('.jpg'):
            if 'GroundTruth' not in root and 'segmentation' not in file.lower():
                image_files.append(os.path.join(root, file))

mask_files = []
for root, dirs, files in os.walk('isic_temp'):
    for file in files:
        if file.lower().endswith('.png'):
            if 'GroundTruth' in root or 'segmentation' in file.lower():
                mask_files.append(os.path.join(root, file))

mask_dict = {}
for m in mask_files:
    base = os.path.basename(m).replace('_segmentation', '').replace('_mask', '').replace('.png', '').replace('.PNG', '')
    mask_dict[base] = m

print(f"Found {len(image_files)} training images. Transferring (limit 500 for colab memory/time)...")
found_pairs = 0
for img_path in image_files:
    base = os.path.basename(img_path).replace('.jpg', '').replace('.JPG', '')
    if base in mask_dict:
        shutil.copy(img_path, os.path.join(dest_dir, 'images', f'{base}.png'))
        shutil.copy(mask_dict[base], os.path.join(dest_dir, 'masks', f'{base}_mask.png'))
        found_pairs += 1
        if found_pairs >= 500:
            break

print(f"Successfully paired and formatted {found_pairs} ISIC 2018 images for the framework!")

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
unzip:  cannot find or open isic-2018-task-1.zip, isic-2018-task-1.zip.zip or isic-2018-task-1.zip.ZIP.
Found 0 training images. Transferring (limit 500 for colab memory/time)...
Successfully paired and formatted 0 ISIC 2018 images for the framework!


In [4]:
# 4. Unified Configuration Injection (For Apples-to-Apples Testing)
import yaml

def update_config(path, epochs=15):
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
    config['training']['epochs'] = epochs
    config['dataset']['name'] = 'medical-image-mask'
    config['dataset']['data_dir'] = 'data/medical_dataset'
    config['dataset']['batch_size'] = 2
    with open(path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

update_config("configs/medical_segmentation.yaml", epochs=20)
update_config("configs/moe_segmentation.yaml", epochs=20)

print("Configurations unified! Training set to 20 epochs.")

Configurations unified! Training set to 20 epochs.


In [5]:
# 5. Train SegMoTE (Foundation Model)
!PYTHONPATH=. python training/train_segmote.py --config configs/medical_segmentation.yaml

2026-09-02 17:11:45.613397: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-02 17:11:58] [INFO] [segmote] - Loaded config from configs/medical_segmentation.yaml
INFO:segmote:Loaded config from configs/medical_segmentation.yaml
[2026-09-02 17:11:58] [INFO] [segmote] - Using device: cuda
INFO:segmote:Using device: cuda
[2026-09-02 17:11:58] [INFO] [segmote] - Initializing datasets and dataloaders...
INFO:segmote:Initializing datasets and dataloaders...
[2026-09-02 17:12:00] [INFO] [segmote] - Loading SegMoTE (SAM + MoTE + PPT)...
INFO:segmote:Loading SegMoTE (SAM + MoTE + PPT)...
[2026-09-02 17:12:02] [INFO] [segmote] - Total params: 94,207,312 | Trainable: 4,530,180
INFO:segmote:Total params: 94,207,312 | Trainable: 4,530,180
/content/vis

In [6]:
# 6. Train MoE-Segformer (Baseline)
!PYTHONPATH=. python training/train_moe.py --config configs/moe_segmentation.yaml

2026-09-02 17:59:23.949880: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-02 17:59:31] [INFO] [moe_segmentation] - Loaded config from configs/moe_segmentation.yaml
INFO:moe_segmentation:Loaded config from configs/moe_segmentation.yaml
[2026-09-02 17:59:31] [INFO] [moe_segmentation] - Using device: cuda
INFO:moe_segmentation:Using device: cuda
[2026-09-02 17:59:31] [INFO] [moe_segmentation] - Initializing datasets and dataloaders...
INFO:moe_segmentation:Initializing datasets and dataloaders...
[2026-09-02 17:59:33] [INFO] [moe_segmentation] - Loading backbone SegFormer: nvidia/mit-b0...
INFO:moe_segmentation:Loading backbone SegFormer: nvidia/mit-b0...
Some weights of SegformerForSemanticSegmentation were not initialized from the model

In [7]:
# 7. Automated Metric Parser & Comparison
import os
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def get_best_metric(log_dir, tag='Metrics/mIoU'):
    best_val = 0.0
    if not os.path.exists(log_dir): return best_val
    for file in os.listdir(log_dir):
        if file.startswith("events.out.tfevents"):
            ea = EventAccumulator(os.path.join(log_dir, file))
            ea.Reload()
            if tag in ea.Tags()['scalars']:
                vals = [e.value for e in ea.Scalars(tag)]
                if max(vals) > best_val: best_val = max(vals)
    return best_val

segmote_iou = get_best_metric('outputs/medical_segmentation/logs', 'Metrics/mIoU')
segmote_dice = get_best_metric('outputs/medical_segmentation/logs', 'Metrics/mDice')

baseline_iou = get_best_metric('outputs/moe_segmentation/logs', 'Metrics/mIoU')
baseline_dice = get_best_metric('outputs/moe_segmentation/logs', 'Metrics/mDice')

df = pd.DataFrame({
    'Model': ['MoE-Segformer (Baseline)', 'SegMoTE (Foundation Model)'],
    'ISIC 2018 Val mIoU': [f"{baseline_iou*100:.2f}%", f"{segmote_iou*100:.2f}%"],
    'ISIC 2018 Val mDice': [f"{baseline_dice*100:.2f}%", f"{segmote_dice*100:.2f}%"],
    'Total Parameters': ['8.07 Million', '94.20 Million']
})

print("\n================ ISIC 2018 COMPARISON RESULTS ================")
display(df)
print("================================================================")


================ ISIC 2018 COMPARISON RESULTS ================


,Model,ISIC 2018 Val mIoU,ISIC 2018 Val mDice,Total Parameters
0,MoE-Segformer (Baseline),65.12%,75.23%,8.07 Million
1,SegMoTE (Foundation Model),65.99%,74.07%,94.20 Million


In [8]:
import shutil
from google.colab import files

# Zip the entire outputs directory containing logs, plots, and checkpoints for both models
print("Zipping the results...")
shutil.make_archive('training_results', 'zip', '/content/vision_tranformer_moe/outputs')

# Trigger the download to your local machine
print("Downloading training_results.zip...")
files.download('training_results.zip')


Zipping the results...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>